# A24 two-point analysis

Standard and GEVP-projected nucleon and $N\sigma$ correlators, multi-state fits, and Laplace checks.

In [1]:
from pathlib import Path
import os
import sys
import warnings

import matplotlib as mpl
mpl.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

HERE = Path.cwd().resolve()
if HERE.parent.name == "__codex_ignore":
    HERE = HERE.parent.parent / HERE.name
if HERE.name != "cA211.530.24" or HERE.parent.name != "07_Nsgm":
    raise RuntimeError("Launch from the cA211.530.24 directory.")
WORK = HERE.parent / "__codex_ignore" / HERE.name
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(HERE.parent))
import util as yu
import util_codex as yuc

yu.setpath("analysis_2pt_codex")
ENS = "a24"
A, AINV = yu.ens2a[ENS], yu.ens2aInv[ENS] / 1000
DELTA = 2
SELECTED = {"N": ("standard", 5), "Nsgm": ("GEVP", 3)}
yuc.apply_paper_style()

GEVP_DELTA = 4
GEVP_WINDOW = (11, 14)


## Correlators and GEVP

Use the eigenvector plateau of the light-current analysis for both projected channels.

In [2]:
c2pt_matrix, _, _ = yu.load_pkl(HERE / "pkl/processData/reg_ignore/data.pkl")
c2pt_matrix = c2pt_matrix.real
times = np.arange(GEVP_DELTA + 1, min(22, c2pt_matrix.shape[1]))
reference_times = times - GEVP_DELTA
eigenvectors = np.array([yu.GEVP(c, reference_times, tList=times)[1] for c in c2pt_matrix])
inverse = np.linalg.inv(eigenvectors)
components = {
    "v": eigenvectors[:, :, 0, 1].real / eigenvectors[:, :, 0, 0].real,
    "s": eigenvectors[:, :, 1, 0].real / eigenvectors[:, :, 1, 1].real,
    "w": 1 / (eigenvectors[:, :, 0, 0].real * inverse[:, :, 0, 0].real) - 1,
}
plateau = (times >= GEVP_WINDOW[0]) & (times <= GEVP_WINDOW[1])
weights = {name: yu.doFit_const(values[:, plateau])[0][:, 0] for name, values in components.items()}
GEVP_TAG = yuc.sample_cache_tag(c2pt_matrix, weights["v"], weights["s"], weights["w"])
vectors = {"N": np.stack([np.ones_like(weights["v"]), weights["v"]], axis=1),
           "Nsgm": np.stack([weights["s"], np.ones_like(weights["s"])], axis=1)}
correlators, effective_masses = {}, {}
for channel, index in [("N", 0), ("Nsgm", 1)]:
    correlators[channel, "standard"] = c2pt_matrix[:, :, index, index]
    vector = vectors[channel]
    correlators[channel, "GEVP"] = np.einsum("ni,ntij,nj->nt", vector, c2pt_matrix, vector)
for key, correlator in correlators.items():
    effective_masses[key] = np.log(correlator[:, :-1] / correlator[:, 1:])
print(f"Eigenvector plateau t/a={GEVP_WINDOW}, t-t0={GEVP_DELTA}a:", {k: yu.jackme_un2str(v) for k, v in weights.items()})
print("Exclusive fit upper bounds:", {k: yu.find_fitmax(v) for k, v in effective_masses.items()})

yuc.guard_fit_cache(c2pt_matrix, weights["v"], weights["s"], weights["w"])


Eigenvector plateau t/a=(11, 14), t-t0=4a: {'v': '0.121(15)', 's': '-0.3929(72)', 'w': '0.0473(67)'}
Exclusive fit upper bounds: {('N', 'standard'): 14, ('N', 'GEVP'): 14, ('Nsgm', 'standard'): 12, ('Nsgm', 'GEVP'): 11}


## Nucleon fits

In [3]:
fits = {}
# The upper bound uses the same effective-mass precision criterion as on B64.
for method in ["standard", "GEVP"]:
    key = ("N", method)
    mass = effective_masses[key]
    upper = yu.find_fitmax(mass)
    starts = [.45, .3, .8, .8, 1]
    ranges = [range(2, upper - 1), range(2, min(upper - 3, 12)), range(1, 4)]
    fits[key] = yuc.fit_meff_comparison(
        mass, ranges[:2], starts.copy(), corrQ=True,
        label=f"N_{method}_one_two_state_v2_codex" + ("_" + GEVP_TAG if method == "GEVP" else ""), overwrite=False,
    )
    if method == "standard":
        seed = [.55, .3, .7] if ENS == "a24" else [.45, .3, .8]
        fits[key][1] = yu.doFits_2pt(mass, ranges[1], yu.func_meff_2st, seed,
                                    label="N_standard_reference_v1_codex", overwrite=False)
    for states, scan in enumerate(fits[key], 1):
        for lower, parameters, chi2, ndof in scan:
            energy = yu.jackme_un2str(parameters[:, 0] * AINV)
            gap = yu.jackme_un2str(parameters[:, 1] * AINV) if states > 1 else "-"
            print("N", method, states, lower, "E0", energy, "gap", gap,
                  "chi2/dof", round(float(np.mean(chi2)) / ndof, 2))


N standard 1 2 E0 1.29911(75) gap - chi2/dof 4221.5
N standard 1 3 E0 1.26304(86) gap - chi2/dof 1123.65
N standard 1 4 E0 1.24352(97) gap - chi2/dof 333.7
N standard 1 5 E0 1.2320(11) gap - chi2/dof 104.3
N standard 1 6 E0 1.2254(12) gap - chi2/dof 41.72
N standard 1 7 E0 1.2204(13) gap - chi2/dof 17.22
N standard 1 8 E0 1.2171(15) gap - chi2/dof 9.67
N standard 1 9 E0 1.2142(17) gap - chi2/dof 5.63
N standard 1 10 E0 1.2113(19) gap - chi2/dof 1.75
N standard 1 11 E0 1.2095(22) gap - chi2/dof 1.09
N standard 1 12 E0 1.2078(27) gap - chi2/dof 0.94
N standard 2 2 E0 1.2196(13) gap 1.378(15) chi2/dof 6.64
N standard 2 3 E0 1.2153(16) gap 1.231(27) chi2/dof 3.07
N standard 2 4 E0 1.2123(20) gap 1.096(52) chi2/dof 2.3
N standard 2 5 E0 1.2090(28) gap 0.913(89) chi2/dof 1.74
N standard 2 6 E0 1.2044(50) gap 0.68(15) chi2/dof 1.48
N standard 2 7 E0 1.184(23) gap 0.30(15) chi2/dof 0.6
N standard 2 8 E0 1.185(32) gap 0.30(25) chi2/dof 0.8
N standard 2 9 E0 1.2037(94) gap 0.91(85) chi2/dof 0.56

## N-sigma fits

In [4]:
# The upper bound uses the same effective-mass precision criterion as on B64.
fits["Nsgm", "standard"] = []
for method in ["GEVP"]:
    key = ("Nsgm", method)
    mass = effective_masses[key]
    upper = yu.find_fitmax(mass)
    starts = [.8, .4, 1, .8, 1]
    ranges = [range(2, upper - 1), range(2, min(upper - 3, 8)), range(1, 4)]
    fits[key] = yuc.fit_meff_comparison(
        mass, ranges[:2], starts.copy(), corrQ=True,
        label=f"Nsgm_{method}_multistate_v1_codex" + ("_" + GEVP_TAG if method == "GEVP" else ""), overwrite=False,
    )
    for states, scan in enumerate(fits[key], 1):
        for lower, parameters, chi2, ndof in scan:
            energy = yu.jackme_un2str(parameters[:, 0] * AINV)
            gap = yu.jackme_un2str(parameters[:, 1] * AINV) if states > 1 else "-"
            print("Nsgm", method, states, lower, "E0", energy, "gap", gap,
                  "chi2/dof", round(float(np.mean(chi2)) / ndof, 2))


Nsgm GEVP 1 2 E0 2.2080(56) gap - chi2/dof 158.19
Nsgm GEVP 1 3 E0 2.0896(75) gap - chi2/dof 29.42
Nsgm GEVP 1 4 E0 2.017(10) gap - chi2/dof 3.88
Nsgm GEVP 1 5 E0 1.981(14) gap - chi2/dof 1.28
Nsgm GEVP 1 6 E0 1.957(20) gap - chi2/dof 0.8
Nsgm GEVP 1 7 E0 1.948(31) gap - chi2/dof 1.02
Nsgm GEVP 1 8 E0 1.929(48) gap - chi2/dof 1.41
Nsgm GEVP 1 9 E0 1.829(76) gap - chi2/dof 0.51
Nsgm GEVP 2 2 E0 1.928(20) gap 1.544(81) chi2/dof 0.79
Nsgm GEVP 2 3 E0 1.945(29) gap 1.77(37) chi2/dof 0.84
Nsgm GEVP 2 4 E0 1.907(93) gap 1.02(90) chi2/dof 0.9
Nsgm GEVP 2 5 E0 0.6(2.3) gap 1.4(2.3) chi2/dof 0.92
Nsgm GEVP 2 6 E0 -1.6(2.3) gap 3.5(2.3) chi2/dof 0.24
Nsgm GEVP 2 7 E0 -1.1(2.5) gap 3.1(2.5) chi2/dof 0.41


## Two-point references

Use the standard nucleon at $t_{\rm low}/a=5$ and the projected $N\sigma$ at $t_{\rm low}/a=3$. Energy differences retain their sample correlations.

In [5]:
selected = {channel: next(fit for fit in fits[channel, method][1] if fit[0] == lower)
            for channel, (method, lower) in SELECTED.items()}
selected_samples = {channel: fit[1] for channel, fit in selected.items()}
references = {
    "nucleon": selected_samples["N"],
    "nsigma": selected_samples["Nsgm"],
    "nsigma_gap": selected_samples["Nsgm"][:, 0] - selected_samples["N"][:, 0],
    "gevp_delta": GEVP_DELTA, "gevp_window": GEVP_WINDOW, "weights": weights,
}
yu.save_pkl_reg("two_point_references", references)
for channel, fit in selected.items():
    pars = fit[1]
    print(channel, SELECTED[channel], "E0 =", yu.jackme_un2str(pars[:, 0] * AINV),
          "E1 =", yu.jackme_un2str(pars[:, :2].sum(axis=1) * AINV))
print("Projected Nsigma - nucleon =", yu.jackme_un2str(references["nsigma_gap"] * AINV), "GeV")


N ('standard', 5) E0 = 1.2090(28) E1 = 2.122(92)
Nsgm ('GEVP', 3) E0 = 1.945(29) E1 = 3.72(39)
Projected Nsigma - nucleon = 0.736(29) GeV


## Laplace checks

In [6]:
# The covariance is recomputed for each trial filter energy and each resample.
def laplace_fit(channel, lower):
    method, _ = SELECTED[channel]
    correlator = correlators[channel, method]
    upper = yu.find_fitmax(effective_masses[channel, method]) - 2 * DELTA
    times_fit = np.arange(lower, upper)
    energy0, gap = np.mean(selected_samples[channel][:, :2], axis=0)
    bounds = np.array([energy0 + .05 / AINV, 5.5 / AINV])
    center, width = bounds.mean(), np.diff(bounds)[0] / 2
    energy = lambda q: center + width * np.tanh(q)
    q0 = np.arctanh((energy0 + gap - center) / width)

    def filtered_mass(q):
        data = 2 * np.cosh(DELTA * energy(q)) * correlator[:, DELTA:-DELTA]
        data = data - correlator[:, 2 * DELTA:] - correlator[:, :-2 * DELTA]
        return np.log(data[:, :-1] / data[:, 1:])

    label = f"{channel}_laplace_lower{lower}_v1_codex" + ("_" + GEVP_TAG if channel == "Nsgm" else "")
    fit = yu.load_pkl_internal(label)
    if fit is None:
        with warnings.catch_warnings(record=True) as messages:
            pars, chi2, ndof, nwarn = yu.jackfit(
                lambda p: np.full(len(times_fit), p[0]),
                lambda q: filtered_mass(q)[:, times_fit], [energy0, q0], maxfev=500,
            )
        valid = np.isfinite(pars).all() and not nwarn
        energies = energy(pars[:, 1])
        valid &= np.all((energies > bounds[0] + .01) & (energies < bounds[1] - .01))
        pars[:, 1] = energies - pars[:, 0]
        fit = (lower, pars, chi2, ndof, bool(valid), int(nwarn))
        yu.save_pkl_internal(label, fit)
    print(channel, "Laplace", lower, "accepted", fit[4], "warnings", fit[5],
          "E0", yu.jackme_un2str(fit[1][:, 0] * AINV),
          "E1", yu.jackme_un2str(fit[1][:, :2].sum(axis=1) * AINV))
    return fit

laplace = {}


In [7]:
laplace["N"] = [laplace_fit("N", lower) for lower in [3, 4, 5]]


N Laplace 3 accepted True warnings 0 E0 1.2151(17) E1 2.428(34)
N Laplace 4 accepted True warnings 0 E0 1.2122(21) E1 2.286(57)
N Laplace 5 accepted True warnings 0 E0 1.2099(29) E1 2.14(11)


In [8]:
laplace["Nsgm"] = [laplace_fit("Nsgm", lower) for lower in [1, 2, 3]]


Nsgm Laplace 1 accepted True warnings 0 E0 1.918(18) E1 3.387(55)
Nsgm Laplace 2 accepted True warnings 0 E0 1.948(24) E1 3.67(21)
Nsgm Laplace 3 accepted True warnings 0 E0 1.946(34) E1 3.63(63)


## Two-point comparison

Open symbols denote standard correlators, filled symbols GEVP projections. Dashed lines mark the two-state reference windows. Late windows with poorly constrained excited states are excluded from the display.

In [9]:
# Publication figures are drawn below with the shared B64-style renderers.


## Additional analysis for the B64-style review

The original two-point support and GEVP plateau are retained. Displacement checks use identical original timeslices for all three displacements. Warning-producing fits are reported, not treated as measurements.

In [10]:
component_scans = {name: [] for name in components}
component_lowers = np.arange(6, GEVP_WINDOW[1] - 1)
for name, data in components.items():
    for lower in component_lowers:
        window = (times >= lower) & (times <= GEVP_WINDOW[1])
        pars, chi2, ndof = yu.doFit_const(data[:, window])
        component_scans[name].append((lower, pars[:, 0], chi2, ndof))



## B64-style two-point figures

The same five layouts are shown separately. Finite but poorly determined late-window fits remain in the notebook's numerical output; the plots stop before their errors obscure the resolved scan.

In [11]:
PLOT_STYLE = yuc.paper_style({"lines.markersize": 3.3, "errorbar.capsize": 2.5})

def display_scan(channel, method, state):
    if state >= len(fits[channel, method]):
        return []
    scan = fits[channel, method][state]
    last = (8 if ENS == "a24" else 11) if channel == "N" else (
        4 if ENS == "a24" and method == "GEVP" else 5)
    return [fit for fit in scan if np.isfinite(fit[1]).all()
            and (state != 1 or fit[0] <= last) and (state != 2 or fit[0] >= 2)]

def plot_correlator_comparison(channel):
    methods = ["standard", "GEVP"]
    masses = [effective_masses[channel, method] for method in methods]
    scans = [[display_scan(channel, method, state) for state in range(len(fits[channel, method]))]
             for method in methods]
    config = {"display_times": [np.arange(1, min(yu.find_fitmax(m) + 1, m.shape[1])) for m in masses]}
    if channel == "Nsgm":
        config["display_times"] = [np.arange(1, m.shape[1]) for m in masses]
        config["effective"] = dict(xlim=(0, 15 * A), ylim=(1.0, 2.9), yticks=[1.0, 1.5, 2.0, 2.5])
        config["effective_legend"] = dict(loc="lower center", bbox_to_anchor=(.5, 1.02))
    chosen = next(f for f in laplace[channel] if f[0] == (5 if channel == "N" else 2))
    if not chosen[4]:
        raise ValueError(f"Unresolved Laplace fit cannot supply the plotted filter: {channel}")
    data = correlators[channel, SELECTED[channel][0]]
    energy = chosen[1][:, :2].sum(axis=1)
    filtered = 2 * np.cosh(DELTA * energy[:, None]) * data[:, DELTA:-DELTA]
    filtered -= data[:, 2 * DELTA:] + data[:, :-2 * DELTA]
    mass = np.log(filtered[:, :-1] / filtered[:, 1:])
    t = np.arange(2, yu.find_fitmax(effective_masses[channel, SELECTED[channel][0]]) - 2 * DELTA)
    lap = (t, mass[:, t], [f for f in laplace[channel] if f[4]])
    selection = (methods.index(SELECTED[channel][0]), SELECTED[channel][1])
    yuc.plot_correlator_comparison(channel, masses, scans, selection, selected_samples[channel], lap, A, AINV, config)

def plot_eigenvectors():
    config = {"window": GEVP_WINDOW, "display": (5, 14), "rows": [dict(ylim=(.025,.215), yticks=[.05,.10,.15,.20]), dict(ylim=(-.425,-.315), yticks=[-.42,-.38,-.34]), dict(ylim=(-.005,.10), yticks=[0,.04,.08])]}
    yuc.plot_eigenvectors(times, components, component_scans, weights, A, config)

def plot_overlap_ratios():
    scans = {(channel, state): [f for f in display_scan(channel, SELECTED[channel][0], state)
                               if yu.jackme(f[1][:, 2])[1] < 2]
             for channel, state in [("N", 1), ("N", 2), ("Nsgm", 1)]}
    yuc.plot_overlap_ratios(scans, {c: s[1] for c, s in SELECTED.items()}, A, {})



In [12]:
with mpl.rc_context(PLOT_STYLE):
    plot_correlator_comparison("N")
    plot_correlator_comparison("Nsgm")
    plot_overlap_ratios()
    plot_eigenvectors()
